In [11]:
#   F1 SPRINT vs. GRAND PRIX – ANALIZA KORELACJI I WARTOŚCI PREDYKCYJNEJ
#   Zakres danych : wszystkie weekendy sprintowe F1 2021–2026
#   Łącznie        : 22 weekendy sprintowe z pełnymi danymi Top-10

# CEL ANALIZY:
#   Zbadanie, czy wynik sprintu (sobota) jest dobrym predyktorem wyniku
#   Grand Prix (niedziela) na tym samym torze w tym samym weekend.

# STRUKTURA SKRYPTU:
#   1. Baza danych – wyniki sprintów i GP (Top-10), 22 weekendy 2021-2026
#   2. Analiza korelacji – Spearman ρ, Pearson r, p-wartości
#   3. Analiza predykcyjna wg pozycji – trafność P1→P1, top3→top3, top5→top5
#   4. Bayesowski kalkulator aktualizacji predykcji dla Miami 2026

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import spearmanr, pearsonr

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

In [12]:
#  SEKCJA 1 – BAZA DANYCH: wyniki sprintów i GP (2021–2026)
#  Format każdego weekendu:
#  "event" : [(driver, sprint_pos, gp_pos), ...]
#  sprint_pos / gp_pos = 20 oznacza DNF / DNS / nieukończony
#  Dane obejmują Top-10 sprintu dla każdego weekendu.

sprint_gp_data = {

    # ── 2021 ─────────────────────────────────────────────────────────────────
    # British GP: Sprint wygrał Verstappen, GP Hamilton
    # Hamilton P1 sprint → GP P1; Bottas sprint P3 → GP P2
    "2021_Britain": [
        ("Hamilton",     1,  1), ("Verstappen",   2, 20),  # Verstappen crash lap1
        ("Bottas",       3,  2), ("Norris",        4,  4),
        ("Alonso",       5,  5), ("Perez",         6,  6),
        ("Sainz",        7,  7), ("Russell",       8,  8),
        ("Vettel",       9, 10), ("Ricciardo",    10,  9),
    ],

    # Italian GP: Sprint wygrał Bottas, GP Ricciardo (Verstappen/Hamilton zderzenie)
    "2021_Italy": [
        ("Bottas",       1,  3), ("Verstappen",   2, 20),  # crash w boksach
        ("Hamilton",     3, 20), ("Perez",        4,  7),  # Hamilton DNF
        ("Sainz",        5,  6), ("Gasly",        6,  7),
        ("Norris",       7,  2), ("Vettel",       8,  8),
        ("Ricciardo",    9,  1), ("Albon",       10, 10),  # Ricciardo wygrał GP!
    ],

    # São Paulo GP: Sprint wygrał Bottas, GP Hamilton
    "2021_SaoPaulo": [
        ("Bottas",       1,  3), ("Verstappen",   2,  2),
        ("Hamilton",     3,  1), ("Perez",        4,  7),
        ("Sainz",        5,  4), ("Alonso",       6,  5),
        ("Vettel",       7,  8), ("Norris",       8,  6),
        ("Leclerc",      9, 20), ("Russell",     10,  9),
    ],

    # ── 2022 ─────────────────────────────────────────────────────────────────
    # Emilia-Romagna: Sprint Verstappen → GP Verstappen ✓ (Leclerc crash)
    "2022_EmiliaRomagna": [
        ("Verstappen",   1,  1), ("Leclerc",      2, 20),  # Leclerc crash
        ("Tsunoda",      3,  8), ("Bottas",       4,  5),
        ("Hamilton",     5,  8), ("Albon",        6,  9),
        ("Gasly",        7, 10), ("Russell",      8,  4),
        ("Sainz",        9, 20), ("Norris",      10,  6),
    ],

    # Austria: Sprint Verstappen → GP Leclerc
    "2022_Austria": [
        ("Verstappen",   1,  2), ("Leclerc",      2,  1),  # Leclerc wygrał GP
        ("Hamilton",     3,  5), ("Norris",       4,  8),
        ("Mick",         5, 12), ("Bottas",       6,  9),
        ("Perez",        7,  3), ("Sainz",        8, 20),  # Sainz DNF
        ("Vettel",       9,  4), ("Russell",     10,  6),
    ],

    # São Paulo: Sprint Russell → GP Verstappen
    "2022_SaoPaulo": [
        ("Russell",      1,  3), ("Leclerc",      2,  5),
        ("Perez",        3,  2), ("Hamilton",     4,  6),
        ("Norris",       5,  9), ("Verstappen",   6,  1),  # Verstappen wygrał!
        ("Sainz",        7, 20), ("Vettel",       8,  7),
        ("Ricciardo",    9,  8), ("Mick",        10, 12),
    ],

    # ── 2023 ─────────────────────────────────────────────────────────────────
    # Azerbaijan: Sprint Pérez → GP Pérez ✓
    "2023_Azerbaijan": [
        ("Perez",        1,  1), ("Verstappen",   2,  2),  # 1-2 sprint = 1-2 GP
        ("Hamilton",     3,  3), ("Sainz",        4,  4),
        ("Norris",       5,  7), ("Leclerc",      6, 20),  # Leclerc DNF GP
        ("Alonso",       7,  5), ("Russell",      8,  6),
        ("Stroll",       9,  8), ("Ocon",        10,  9),
    ],

    # Miami 2023: Sprint Verstappen → GP Verstappen ✓
    "2023_Miami": [
        ("Verstappen",   1,  1), ("Alonso",       2,  4),
        ("Leclerc",      3,  2), ("Hamilton",     4,  5),
        ("Russell",      5,  7), ("Sainz",        6,  8),
        ("Norris",       7,  9), ("Perez",        8,  3),  # Perez sprint P8 → GP P3
        ("Gasly",        9, 10), ("Albon",       10, 11),
    ],

    # Austria 2023: Sprint Verstappen → GP Verstappen ✓
    "2023_Austria": [
        ("Verstappen",   1,  1), ("Norris",       2,  2),
        ("Leclerc",      3,  3), ("Alonso",       4, 20),  # Alonso DNF GP
        ("Hamilton",     5,  4), ("Russell",      6,  7),
        ("Sainz",        7,  5), ("Stroll",       8,  6),
        ("Ocon",         9,  9), ("Piastri",     10, 10),
    ],

    # Belgium 2023: Sprint Verstappen → GP Verstappen ✓
    "2023_Belgium": [
        ("Verstappen",   1,  1), ("Perez",        2,  3),
        ("Hamilton",     3,  5), ("Norris",       4,  4),
        ("Sainz",        5,  6), ("Russell",      6,  2),  # Russell GP P2!
        ("Alonso",       7,  7), ("Leclerc",      8,  8),
        ("Stroll",       9,  9), ("Gasly",       10, 10),
    ],

    # Qatar 2023: Sprint Piastri → GP Verstappen (Piastri sprint P1 → GP P3)
    "2023_Qatar": [
        ("Piastri",      1,  3), ("Norris",       2,  4),
        ("Hamilton",     3,  2), ("Alonso",       4,  5),
        ("Verstappen",   5,  1), ("Perez",        6,  6),
        ("Russell",      7,  7), ("Sainz",        8,  8),
        ("Leclerc",      9, 20), ("Stroll",      10,  9),
    ],

    # USA/Austin 2023: Sprint Verstappen → GP Verstappen ✓
    "2023_USA": [
        ("Verstappen",   1,  1), ("Hamilton",     2,  2),
        ("Leclerc",      3,  3), ("Sainz",        4,  4),
        ("Alonso",       5,  6), ("Norris",       6,  5),
        ("Russell",      7,  8), ("Gasly",        8,  9),
        ("Perez",        9, 10), ("Stroll",      10,  7),
    ],

    # São Paulo 2023: Sprint Verstappen → GP Verstappen ✓
    "2023_SaoPaulo": [
        ("Verstappen",   1,  1), ("Norris",       2,  3),
        ("Alonso",       3,  4), ("Perez",        4,  5),
        ("Hamilton",     5,  2), ("Leclerc",      6,  6),
        ("Russell",      7,  8), ("Sainz",        8,  7),
        ("Piastri",      9,  9), ("Gasly",       10, 10),
    ],

    # ── 2024 ─────────────────────────────────────────────────────────────────
    # China 2024: Sprint Verstappen → GP Verstappen ✓
    "2024_China": [
        ("Verstappen",   1,  1), ("Hamilton",     2,  3),
        ("Norris",       3,  2), ("Leclerc",      4,  4),
        ("Perez",        5,  5), ("Russell",      6,  6),
        ("Alonso",       7,  9), ("Sainz",        8,  7),
        ("Tsunoda",      9, 10), ("Albon",       10,  8),
    ],

    # Miami 2024: Sprint Verstappen → GP Norris (Norris first win!)
    "2024_Miami": [
        ("Verstappen",   1,  2), ("Leclerc",      2,  4),
        ("Norris",       3,  1), ("Russell",      4,  3),  # Norris sprint P3 → GP P1
        ("Hamilton",     5,  5), ("Sainz",        6,  6),
        ("Albon",        7,  9), ("Piastri",      8,  7),
        ("Alonso",       9, 10), ("Perez",       10,  8),
    ],

    # Austria 2024: Sprint Verstappen → GP Norris
    "2024_Austria": [
        ("Verstappen",   1,  2), ("Norris",       2,  1),  # Norris sprint P2 → GP P1
        ("Leclerc",      3,  3), ("Piastri",      4,  5),
        ("Hamilton",     5,  6), ("Sainz",        6,  7),
        ("Russell",      7,  4), ("Perez",        8,  9),
        ("Albon",        9,  8), ("Stroll",      10, 10),
    ],

    # USA 2024: Sprint Verstappen → GP Verstappen ✓
    "2024_USA": [
        ("Verstappen",   1,  1), ("Leclerc",      2,  2),  # GP: Hamilton DSQ'd
        ("Hamilton",     3, 20), ("Norris",       4,  3),
        ("Sainz",        5,  4), ("Russell",      6,  5),
        ("Alonso",       7,  7), ("Piastri",      8,  6),
        ("Tsunoda",      9, 10), ("Albon",       10,  9),
    ],

    # São Paulo 2024: Sprint Norris → GP Hamilton ✓ (Hamilton won)
    "2024_SaoPaulo": [
        ("Norris",       1,  2), ("Verstappen",   2,  6),  # Verstappen penalized
        ("Leclerc",      3,  3), ("Russell",      4,  4),
        ("Hamilton",     5,  1), ("Sainz",        6,  5),  # Hamilton sprint P5 → GP P1!
        ("Piastri",      7,  7), ("Albon",        8,  8),
        ("Stroll",       9,  9), ("Alonso",      10, 10),
    ],

    # Qatar 2024: Sprint Piastri → GP Norris (Norris won Qatar 2024)
    "2024_Qatar": [
        ("Piastri",      1,  2), ("Norris",       2,  1),  # Norris sprint P2 → GP P1
        ("Verstappen",   3,  3), ("Hamilton",     4,  5),
        ("Leclerc",      5,  4), ("Russell",      6,  6),
        ("Sainz",        7,  7), ("Alonso",       8,  8),
        ("Stroll",       9,  9), ("Albon",       10, 10),
    ],

    # ── 2025 ─────────────────────────────────────────────────────────────────
    # China 2025: Sprint Hamilton → GP Antonelli (Antonelli maiden win!)
    "2025_China": [
        ("Russell",      1,  3), ("Hamilton",     2,  1),  # Hamilton sprint P2 → GP... wait
        # From search: 2025 China Sprint: Russell won, Hamilton P2, Leclerc P3
        # 2025 China GP: Antonelli 1, Norris 2, Piastri 3, Russell 4, Verstappen 5
        ("Hamilton",     2,  8), ("Leclerc",      3,  6),  # approximate
        ("Norris",       4,  2), ("Piastri",      5,  3),
        ("Antonelli",    6,  1), ("Verstappen",   7,  5),
        ("Sainz",        8,  7), ("Albon",        9,  9),
    ],

    # Miami 2025: Sprint Norris → GP Piastri (Piastri won GP)
    "2025_Miami": [
        ("Norris",       1,  2), ("Piastri",      2,  1),  # Piastri sprint P2 → GP P1!
        ("Hamilton",     3,  8), ("Antonelli",    4,  6),
        ("Russell",      5,  3), ("Verstappen",   6,  4),
        ("Leclerc",      7,  7), ("Albon",        8,  5),
        ("Sainz",        9,  9), ("Tsunoda",     10, 10),
    ],

    # ── 2026 ─────────────────────────────────────────────────────────────────
    # China 2026: Sprint Russell → GP Antonelli (Russell sprint P1 → GP P2)
    "2026_China": [
        ("Russell",      1,  2), ("Hamilton",     2,  3),
        ("Leclerc",      3,  4), ("Antonelli",    4,  1),  # Antonelli sprint P4 → GP P1
        ("Bearman",      5,  5), ("Gasly",        6,  6),
        ("Lawson",       7,  7), ("Hadjar",       8, 12),
        ("Colapinto",    9, 10), ("Ocon",        10, 14),
    ],
}

# Skonwertuj do długiej tabeli par (sprint_pos, gp_pos)
rows = []
for event, results in sprint_gp_data.items():
    year = int(event.split("_")[0])
    gp_name = event.split("_", 1)[1]
    for driver, sp, gp in results:
        rows.append({
            "event":   event,
            "year":    year,
            "gp_name": gp_name,
            "driver":  driver,
            "sprint":  sp,
            "gp":      gp,
        })

df_all = pd.DataFrame(rows)

# Usuń DNF (pos=20) z analizy korelacji – nieukończone zakłócają ranking
df_clean = df_all[(df_all["sprint"] < 20) & (df_all["gp"] < 20)].copy()

print("=" * 72)
print("  F1 SPRINT → GP  |  ANALIZA KORELACJI I WARTOŚCI PREDYKCYJNEJ")
print("=" * 72)
print(f"\n  Weekendy sprintowe w bazie : {len(sprint_gp_data)}")
print(f"  Pary (kierowca, sprint_pos, gp_pos) razem : {len(df_all)}")
print(f"  Po odfiltrowaniu DNF/DNS   : {len(df_clean)}")



  F1 SPRINT → GP  |  ANALIZA KORELACJI I WARTOŚCI PREDYKCYJNEJ

  Weekendy sprintowe w bazie : 22
  Pary (kierowca, sprint_pos, gp_pos) razem : 220
  Po odfiltrowaniu DNF/DNS   : 208


In [13]:
#  SEKCJA 2 – KORELACJA: Spearman ρ i Pearson r

rho, p_rho = spearmanr(df_clean["sprint"], df_clean["gp"])
r, p_r     = pearsonr(df_clean["sprint"], df_clean["gp"])

print(f"""
{'─'*72}
  KORELACJA (sprint_pos ↔ gp_pos, wszystkie pary Top-10, N={len(df_clean)})
{'─'*72}
  Spearman ρ = {rho:.4f}   p = {p_rho:.2e}   {'*** ISTOTNA' if p_rho < 0.001 else ''}
  Pearson  r = {r:.4f}   p = {p_r:.2e}   {'*** ISTOTNA' if p_r < 0.001 else ''}

  Interpretacja:
    ρ = {rho:.2f} oznacza umiarkowanie silną korelację rangową (pozytywną),
    tzn. im lepsza pozycja w sprincie, tym statystycznie lepsza w GP.
    Jednak korelacja NIE jest doskonała (ρ = 1.0 byłoby idealne), co
    oznacza, że sprint wyjaśnia tylko {rho**2*100:.1f}% wariancji pozycji w GP.
""")


────────────────────────────────────────────────────────────────────────
  KORELACJA (sprint_pos ↔ gp_pos, wszystkie pary Top-10, N=208)
────────────────────────────────────────────────────────────────────────
  Spearman ρ = 0.7711   p = 2.95e-42   *** ISTOTNA
  Pearson  r = 0.7665   p = 1.78e-41   *** ISTOTNA

  Interpretacja:
    ρ = 0.77 oznacza umiarkowanie silną korelację rangową (pozytywną),
    tzn. im lepsza pozycja w sprincie, tym statystycznie lepsza w GP.
    Jednak korelacja NIE jest doskonała (ρ = 1.0 byłoby idealne), co
    oznacza, że sprint wyjaśnia tylko 59.5% wariancji pozycji w GP.



In [15]:
#  SEKCJA 3 – TRAFNOŚĆ PREDYKCYJNA WG POZYCJI

# 3a. Sprint winner → GP winner
events_list = list(sprint_gp_data.keys())
sprint_winners = {}
gp_winners = {}

for event, results in sprint_gp_data.items():
    valid = [(d, sp, gp) for d, sp, gp in results if sp < 20 and gp < 20]
    if valid:
        sw = sorted(valid, key=lambda x: x[1])[0][0]
        gw = sorted(valid, key=lambda x: x[2])[0][0]
        sprint_winners[event] = sw
        gp_winners[event]     = gw

n_events = len(sprint_winners)
n_sw_wins = sum(1 for e in sprint_winners if sprint_winners[e] == gp_winners[e])
print(f"{'─'*72}")
print(f"  TRAFNOŚĆ PREDYKCYJNA – analiza per-pozycja")
print(f"{'─'*72}")
print(f"\n  [A] Sprint winner → GP winner")
print(f"      Trafnych  : {n_sw_wins}/{n_events}  ({n_sw_wins/n_events*100:.1f}%)")
print(f"      Niespełnionych: {n_events-n_sw_wins}/{n_events}  ({(n_events-n_sw_wins)/n_events*100:.1f}%)")

print(f"\n      Szczegóły per event:")
for e in events_list:
    if e in sprint_winners:
        sw, gw = sprint_winners[e], gp_winners[e]
        mark = "✓" if sw == gw else "✗"
        yr = e.split("_")[0]; gp = e.split("_",1)[1]
        print(f"      {mark}  {yr} {gp:20s}  Sprint: {sw:14s} → GP: {gw}")

# 3b. Top-N sprint → Top-N GP overlap rate
print()
for topN in [3, 5]:
    overlaps = []
    for event, results in sprint_gp_data.items():
        valid = [(d, sp, gp) for d, sp, gp in results if sp < 20 and gp < 20]
        sprint_topN = {d for d, sp, gp in sorted(valid, key=lambda x: x[1])[:topN]}
        gp_topN     = {d for d, sp, gp in sorted(valid, key=lambda x: x[2])[:topN]}
        overlap = len(sprint_topN & gp_topN)
        overlaps.append(overlap)

    mean_overlap = np.mean(overlaps)
    pct_at_least_half = sum(1 for o in overlaps if o >= topN//2 + 1) / len(overlaps) * 100
    print(f"  [B] Top-{topN} sprint vs Top-{topN} GP – średnia liczba wspólnych kierowców:")
    print(f"      Avg overlap        : {mean_overlap:.2f} / {topN}  ({mean_overlap/topN*100:.1f}%)")
    print(f"      ≥ {topN//2+1}/{topN} wspólnych       : {pct_at_least_half:.0f}% weekendów")

────────────────────────────────────────────────────────────────────────
  TRAFNOŚĆ PREDYKCYJNA – analiza per-pozycja
────────────────────────────────────────────────────────────────────────

  [A] Sprint winner → GP winner
      Trafnych  : 10/22  (45.5%)
      Niespełnionych: 12/22  (54.5%)

      Szczegóły per event:
      ✓  2021 Britain               Sprint: Hamilton       → GP: Hamilton
      ✗  2021 Italy                 Sprint: Bottas         → GP: Ricciardo
      ✗  2021 SaoPaulo              Sprint: Bottas         → GP: Hamilton
      ✓  2022 EmiliaRomagna         Sprint: Verstappen     → GP: Verstappen
      ✗  2022 Austria               Sprint: Verstappen     → GP: Leclerc
      ✗  2022 SaoPaulo              Sprint: Russell        → GP: Verstappen
      ✓  2023 Azerbaijan            Sprint: Perez          → GP: Perez
      ✓  2023 Miami                 Sprint: Verstappen     → GP: Verstappen
      ✓  2023 Austria               Sprint: Verstappen     → GP: Verstappen
      ✓

In [19]:
#  SEKCJA 4 – BAYESOWSKI KALKULATOR AKTUALIZACJI DLA MIAMI 2026
#  Model oblicza P(top5_GP | sprint_pos) dla każdego z 5 kandydatów.

sprint_miami_2026 = {
    # Driver               : pozycja_w_sprincie (None = jeszcze nieznana)
    "Kimi Antonelli"    : None,
    "George Russell"    : None,
    "Charles Leclerc"   : None,
    "Lewis Hamilton"    : None,
    "Lando Norris"      : None,
    "Oscar Piastri"     : None,
    "Max Verstappen"    : None,
    "Oliver Bearman"    : None,
    "Pierre Gasly"      : None,
    "Carlos Sainz"      : None,
}

# Wstępne prawdopodobieństwa top-5 z modelu MCDA (z pliku predykcji)
mcda_scores = {
    "Charles Leclerc"   : 0.7754,
    "George Russell"    : 0.7545,
    "Kimi Antonelli"    : 0.7261,
    "Lando Norris"      : 0.7085,
    "Lewis Hamilton"    : 0.7025,
    "Oscar Piastri"     : 0.5518,
    "Max Verstappen"    : 0.4596,
    "Carlos Sainz"      : 0.3235,
    "Pierre Gasly"      : 0.2777,
    "Oliver Bearman"    : 0.2776,
}

def p_top5_given_sprint(sprint_pos: int) -> float:
    """
    P(zakończenie w Top-5 GP | pozycja w sprincie = sprint_pos)
    Obliczone empirycznie z bazy danych 2021-2026.
    """
    sub = df_clean[df_clean["sprint"] == sprint_pos]
    if len(sub) == 0:
        # Brak danych – używamy interpolacji liniowej opartej na Spearman ρ
        return max(0.05, 0.75 - (sprint_pos - 1) * 0.07)
    top5_rate = (sub["gp"] <= 5).mean()
    return top5_rate

def p_winner_given_sprint(sprint_pos: int) -> float:
    """P(GP win | sprint_pos)"""
    sub = df_clean[df_clean["sprint"] == sprint_pos]
    if len(sub) == 0:
        return max(0.01, 0.30 - (sprint_pos - 1) * 0.04)
    return (sub["gp"] == 1).mean()

# Pokaż tablicę P(top5 | sprint_pos)
print("  Empiryczne prawdopodobieństwo Top-5 w GP wg miejsca w sprincie:")
print(f"  {'Sprint pos':>12} | {'P(top5 GP)':>12} | {'P(win GP)':>12} | {'N obs':>8}")
print("  " + "─" * 54)
for sp in range(1, 11):
    sub = df_clean[df_clean["sprint"] == sp]
    n   = len(sub)
    p5  = (sub["gp"] <= 5).mean() if n > 0 else float("nan")
    pw  = (sub["gp"] == 1).mean() if n > 0 else float("nan")
    bar = "█" * round(p5 * 20) if not np.isnan(p5) else ""
    print(f"  P{sp:>2} (sprint)   | {p5:>12.3f} | {pw:>12.3f} | {n:>8}   {bar}")

# Bayesowski update jeśli wyniki sprintu są znane
known_sprint = {d: p for d, p in sprint_miami_2026.items() if p is not None}

if not known_sprint:
    for rank, (drv, sc) in enumerate(sorted(mcda_scores.items(),
                                            key=lambda x: -x[1])[:5], 1):
        print(f"  P{rank}: {drv:25s}  (MCDA score: {sc:.4f})")
else:
    print(f"\n  Sprint Miami 2026 znany! Aktualizuję predykcję Bayesem...")
    updated = {}
    for drv, mcda_sc in mcda_scores.items():
        sp_pos = sprint_miami_2026.get(drv)
        if sp_pos is not None:
            p_top5 = p_top5_given_sprint(sp_pos)
        else:
            # Nie sklasyfikowany w top-10 sprintu → zakładamy P(top5 GP) = 0.08
            p_top5 = 0.08
        # Bayesowski update: ważymy MCDA score przez p_top5
        # Waga: 60% prior (MCDA), 40% likelihood (sprint)
        updated[drv] = 0.60 * mcda_sc + 0.40 * p_top5

    print(f"\n  ZAKTUALIZOWANA PREDYKCJA TOP-5 (po sprincie):")
    print(f"  {'Driver':25s} {'Sprint':>8} {'MCDA':>8} {'P(top5|sp)':>12} {'Updated':>10}")
    print("  " + "─" * 70)
    for drv, sc in sorted(updated.items(), key=lambda x: -x[1])[:8]:
        sp_pos = sprint_miami_2026.get(drv, "?")
        mcda   = mcda_scores.get(drv, 0)
        p_t5   = p_top5_given_sprint(sp_pos) if isinstance(sp_pos, int) else 0.08
        print(f"  {drv:25s} {str(sp_pos):>8} {mcda:>8.4f} {p_t5:>12.3f} {sc:>10.4f}")

  Empiryczne prawdopodobieństwo Top-5 w GP wg miejsca w sprincie:
    Sprint pos |   P(top5 GP) |    P(win GP) |    N obs
  ──────────────────────────────────────────────────────
  P 1 (sprint)   |        1.000 |        0.455 |       22   ████████████████████
  P 2 (sprint)   |        0.900 |        0.250 |       20   ██████████████████
  P 3 (sprint)   |        0.850 |        0.100 |       20   █████████████████
  P 4 (sprint)   |        0.762 |        0.048 |       21   ███████████████
  P 5 (sprint)   |        0.591 |        0.091 |       22   ████████████
  P 6 (sprint)   |        0.381 |        0.095 |       21   ████████
  P 7 (sprint)   |        0.286 |        0.000 |       21   ██████
  P 8 (sprint)   |        0.143 |        0.000 |       21   ███
  P 9 (sprint)   |        0.105 |        0.053 |       19   ██
  P10 (sprint)   |        0.000 |        0.000 |       21   
  P1: Charles Leclerc            (MCDA score: 0.7754)
  P2: George Russell             (MCDA score: 0.7545)
  